In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tabulate import tabulate 

### Loading the dataset

In [4]:
file_path = 'cleaned_dataset.csv'
movies_df = pd.read_csv(file_path)

### Preprocessing Data

Combining relevant columns into a single textual representation

In [5]:
movies_df['combined_text'] = (
    movies_df['overview'] + ' ' +
    movies_df['genres'] + ' ' +
    movies_df['keywords'])

Initializing TF-IDF Vectorizer with built-in stopwords

In [6]:
tfidf = TfidfVectorizer(stop_words='english')

Fiting and transforming the combined text

In [7]:
tfidf_matrix = tfidf.fit_transform(movies_df['combined_text'])

### Function to recommend movies

In [8]:
def recommend_movies(user_input, tfidf_matrix, tfidf, movies_df, top_n=5):
    user_input_tfidf = tfidf.transform([user_input])

    # Compute cosine similarity with all movies
    cosine_sim = cosine_similarity(user_input_tfidf, tfidf_matrix).flatten()
    top_indices = cosine_sim.argsort()[-top_n:][::-1]

    recommendations = movies_df.iloc[top_indices].copy()
    recommendations['similarity_score'] = cosine_sim[top_indices]

    # Extract titles and scores for separate use
    titles = recommendations['title'].tolist()
    scores = recommendations['similarity_score'].tolist()

    return titles, scores, recommendations

Function to format movie details

In [9]:
def format_movie_details(movie):

    runtime_hours = movie['runtime'] // 60
    runtime_minutes = movie['runtime'] % 60
    formatted_details = (
        f"Title: {movie['title']}\n"
        f"Released Date: {int(movie['release_date']) if not pd.isna(movie['release_date']) else 'Unknown'}\n"
        f"Runtime: {runtime_hours}h {runtime_minutes}m\n"
        f"Genre: {movie['genres']}\n"
        f"Overview: {movie['overview']}\n"
    )
    return formatted_details

### Main program loop

In [10]:
while True:
    user_query = input('Enter keywords (or type 0 to quit): ').strip()
    if user_query == '0':
        print('Goodbye!')
        break
   
    titles, scores, recommendations = recommend_movies(user_query, tfidf_matrix, tfidf, movies_df, top_n=5)
    
    # Display detailed information for each recommendation (optional)
    print('\nRecommended Movies:')
    for _, movie in recommendations.iterrows():
        print(format_movie_details(movie))
        print('-' * 40)
        
    # Display a formatted table of titles and similarity score
    print('-' * 53)
    print(f'| {"Title":<30} | {"Similarity Score":<15} |')
    print('-' * 53)
    for title, score in zip(titles, scores):
        print(f'| {title:<30} | {score:<15.4f}  |')
    print('-' * 53)

Enter keywords (or type 0 to quit):  horror, game, suspense, family , wedding, escape


Recommended Movies:
Title: Ready or Not
Released Date: 2019
Runtime: 1h 35m
Genre: Comedy, Horror, Thriller
Overview: A bride's wedding night takes a sinister turn when her eccentric new in-laws force her to take part in a terrifying game.

----------------------------------------
Title: The Week Of
Released Date: 2018
Runtime: 1h 56m
Genre: Comedy
Overview: The parents of a soon-to-be married couple make the final preparations for the wedding ceremony.

----------------------------------------
Title: KING GAME
Released Date: 2010
Runtime: 1h 33m
Genre: Drama, Thriller, Mystery
Overview: 5 girls and 5 guys find themselves mysteriously trapped in a room and are forced to play a psychological game of dare.

----------------------------------------
Title: Play or Die
Released Date: 2019
Runtime: 1h 29m
Genre: Horror, Mystery
Overview: Welcome to Paranoia, the ultimate escape game. Rule #1: Nothing is real. Rule #2: One of you will die. Lucas and Chloe, two passionate gamers, decide to pa

Enter keywords (or type 0 to quit):  0

Goodbye!
